# 🚀 RAG High-Score Experiment Notebook

## 실험 목표
- **검색 품질 향상**: Sparse / Dense / Hybrid 조합 최적화
- **Reranking 강화**: Fine-tuned Qwen3-Reranker-8B 활용
- **앙상블 고도화**: RRF + DualFuse + 학습형 메타 라우터
- **쿼리 분류 정밀화**: 과학 상식 vs 일반 대화 라우팅

## 실험 구조
```
Phase 0: 환경 확인 & 데이터 검증
Phase 1: 검색 전략 실험 (Sparse / Dense / Hybrid)
Phase 2: Reranker 실험 (fine-tuned vs baseline)
Phase 3: 앙상블 실험 (RRF / DualFuse / 가중 앙상블)
Phase 4: 쿼리 라우터 실험 (Heuristic / Learned)
Phase 5: 제출 파일 생성 & 검증
```

---
## Phase 0: 환경 확인 & 경로 설정

In [ ]:
import os, sys, json, csv, ast, re, math, hashlib, time
from pathlib import Path
from collections import defaultdict, Counter
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# ── 경로 설정 ──────────────────────────────────────────────
ROOT        = Path("/root")
CODE_DIR    = ROOT / "code"
DATA_DIR    = ROOT / "data"
ART_DIR     = CODE_DIR / "artifacts"
SUB_DIR     = ART_DIR / "submissions"
MODEL_DIR   = ART_DIR / "qwen3-reranker-8b-science"

EVAL_PATH   = DATA_DIR / "eval.jsonl"       # 쿼리 파일
GT_PATH     = DATA_DIR / "gt.jsonl"         # 정답 (있을 경우)
DOC_META    = ART_DIR / "doc_metadata.jsonl"
SYN_PATH    = ART_DIR / "science_synonyms.txt"
UDICT_PATH  = ART_DIR / "user_dict.txt"

BASE_SUB    = SUB_DIR / "submission_m4.csv"  # 현재 베이스라인
INDEX_NAME  = "science_rag_m4"               # Elasticsearch 인덱스명
ES_HOST     = "http://localhost:9200"

print("[경로 확인]")
for p in [EVAL_PATH, GT_PATH, DOC_META, MODEL_DIR, BASE_SUB]:
    status = "✅" if p.exists() else "❌"
    print(f"  {status} {p}")

In [ ]:
# ── 공통 유틸 함수 ─────────────────────────────────────────

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def looks_like_jsonl(path: Path) -> bool:
    with path.open("r", encoding="utf-8-sig", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if s:
                return s.startswith("{") and s.endswith("}")
    return False

def read_rows(path: Path) -> Tuple[List[Dict], str]:
    if looks_like_jsonl(path):
        rows = []
        with path.open("r", encoding="utf-8-sig") as f:
            for line in f:
                s = line.strip()
                if s:
                    rows.append(json.loads(s))
        rows = [{k.lstrip('\ufeff').strip(): v for k, v in r.items()} for r in rows]
        return rows, "jsonl"
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    rows = [{k.lstrip('\ufeff').strip(): v for k, v in r.items()} for r in rows]
    return rows, "csv"

def write_rows(path: Path, rows: List[Dict], fmt: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "jsonl":
        with path.open("w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    else:
        with path.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)

def detect_col(keys, candidates, required=True):
    lowered = {k.lower(): k for k in keys}
    for c in candidates:
        if c in keys: return c
    for c in candidates:
        if c in lowered: return lowered[c]
    if required:
        raise KeyError(f"Column not found: {candidates}  keys={list(keys)}")
    return None

def parse_docids(value) -> List[str]:
    if value is None: return []
    if isinstance(value, list):
        out = []
        for x in value:
            if isinstance(x, dict):
                out.append(str(x.get("docid") or x.get("id") or x))
            else:
                out.append(str(x))
        return out
    if isinstance(value, dict):
        return [str(value.get("docid") or value.get("id") or value)]
    s = str(value).strip()
    if not s: return []
    try:
        return parse_docids(ast.literal_eval(s))
    except Exception:
        pass
    if "|" in s: return [x.strip() for x in s.split("|") if x.strip()]
    if "," in s and "[" not in s: return [x.strip() for x in s.split(",") if x.strip()]
    return [s]

# ── MAP@3 평가 함수 ────────────────────────────────────────
def ap_at_3(pred: List[str], gt: List[str]) -> float:
    gt_set = set(gt)
    if not gt_set: return 1.0
    hits, total = 0, 0.0
    for rank, docid in enumerate(pred[:3], start=1):
        if docid in gt_set:
            hits += 1
            total += hits / rank
    return total / min(len(gt_set), 3)

def map_at_3(preds: Dict[str, List[str]], gts: Dict[str, List[str]]) -> float:
    scores = [ap_at_3(preds.get(eid, []), gts.get(eid, [])) for eid in gts]
    return float(np.mean(scores)) if scores else 0.0

print("공통 유틸 로드 완료 ✅")

In [ ]:
# ── eval 데이터 로드 및 구조 확인 ─────────────────────────
eval_rows, eval_fmt = read_rows(EVAL_PATH)
print(f"eval 샘플 수: {len(eval_rows)}, 포맷: {eval_fmt}")
print("컬럼:", list(eval_rows[0].keys()))
print("\n--- 샘플 5개 ---")
for r in eval_rows[:5]:
    print(r)

EVAL_ID_COL = detect_col(list(eval_rows[0].keys()),
                         ["eval_id", "id", "query_id", "qid"])
EVAL_Q_COL  = detect_col(list(eval_rows[0].keys()),
                         ["standalone_query", "query", "question", "user_query", "msg"],
                         required=False)
print(f"\nID 컬럼: {EVAL_ID_COL}, 쿼리 컬럼: {EVAL_Q_COL}")

eval_map = {str(r[EVAL_ID_COL]).strip(): r for r in eval_rows}

def get_query(row: Dict) -> str:
    """eval row에서 쿼리 텍스트 추출"""
    if EVAL_Q_COL and row.get(EVAL_Q_COL):
        return str(row[EVAL_Q_COL]).strip()
    # msg가 JSON인 경우
    msg = row.get("msg", "")
    if isinstance(msg, str):
        try:
            msgs = json.loads(msg)
            if isinstance(msgs, list):
                for m in reversed(msgs):
                    if m.get("role") == "user":
                        return str(m.get("content", "")).strip()
        except Exception:
            return msg.strip()
    return ""

In [ ]:
# ── gt 데이터 로드 (있는 경우) ─────────────────────────────
gt_map: Dict[str, List[str]] = {}
if GT_PATH.exists():
    gt_rows, _ = read_rows(GT_PATH)
    gt_id_col  = detect_col(list(gt_rows[0].keys()),
                            ["eval_id", "id", "query_id", "qid"])
    gt_ref_col = detect_col(list(gt_rows[0].keys()),
                            ["references", "docids", "answer_docids", "topk", "gt"],
                            required=False)
    if gt_ref_col is None:
        gt_ref_col = [k for k in gt_rows[0].keys() if k != gt_id_col][0]
    gt_map = {str(r[gt_id_col]).strip(): parse_docids(r.get(gt_ref_col, []))
              for r in gt_rows}
    print(f"GT 로드: {len(gt_map)}개 ✅")
else:
    print("GT 없음 – MAP@3 내부 평가 불가 (오프라인 검증만 가능)")

# ── 베이스라인 제출 파일 로드 ─────────────────────────────
base_rows, base_fmt = read_rows(BASE_SUB)
BASE_ID_COL  = detect_col(list(base_rows[0].keys()), ["eval_id", "id", "query_id"])
BASE_REF_COL = detect_col(list(base_rows[0].keys()),
                          ["references", "topk", "docids", "pred_docids"])
base_preds = {str(r[BASE_ID_COL]).strip(): parse_docids(r.get(BASE_REF_COL, []))
              for r in base_rows}
print(f"베이스라인 ({BASE_SUB.name}): {len(base_preds)}개 로드")
if gt_map:
    score = map_at_3(base_preds, gt_map)
    print(f"  ↳ MAP@3 (baseline): {score:.4f}")

---
## Phase 1: 검색 전략 실험

### 1-A: Elasticsearch 연결 & 기본 Sparse BM25

In [ ]:
from elasticsearch import Elasticsearch, NotFoundError

es = Elasticsearch(ES_HOST)
info = es.info()
print(f"ES 버전: {info['version']['number']}")
print(f"인덱스 {INDEX_NAME} 존재:", es.indices.exists(index=INDEX_NAME))

# 인덱스 stats
try:
    stats = es.indices.stats(index=INDEX_NAME)
    doc_count = stats["_all"]["primaries"]["docs"]["count"]
    print(f"색인된 문서 수: {doc_count:,}")
except Exception as e:
    print(f"인덱스 stats 에러: {e}")

In [ ]:
# ── 동의어 사전 로드 ───────────────────────────────────────
science_synonyms: List[List[str]] = []
if SYN_PATH.exists():
    with SYN_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            parts = [x.strip() for x in line.strip().split(",") if x.strip()]
            if len(parts) >= 2:
                science_synonyms.append(parts)
    print(f"동의어 그룹 수: {len(science_synonyms)}")
    print("예시:", science_synonyms[:3])

def expand_query_with_synonyms(query: str) -> str:
    """쿼리에 동의어를 추가해 리트리벌 향상"""
    expansions = []
    for group in science_synonyms:
        for term in group:
            if term in query:
                expansions.extend([t for t in group if t != term])
                break
    if expansions:
        return query + " " + " ".join(expansions)
    return query

In [ ]:
# ── Sparse BM25 검색 함수 ──────────────────────────────────
def search_sparse(
    query: str,
    index: str = INDEX_NAME,
    top_k: int = 10,
    use_synonym: bool = True,
    phrase_boost: float = 2.0,
    fields: List[str] = None,
) -> List[Dict]:
    if fields is None:
        fields = ["title^2", "content"]
    q = expand_query_with_synonyms(query) if use_synonym else query

    # match + phrase boost 조합
    dsl = {
        "size": top_k,
        "query": {
            "bool": {
                "should": [
                    {"multi_match": {
                        "query": q,
                        "fields": fields,
                        "type": "best_fields",
                        "operator": "or",
                    }},
                    {"multi_match": {
                        "query": query,  # 원문으로 phrase
                        "fields": fields,
                        "type": "phrase",
                        "boost": phrase_boost,
                    }},
                ]
            }
        },
        "_source": ["docid", "title", "content"],
    }
    resp = es.search(index=index, body=dsl)
    return [
        {"docid": h["_source"].get("docid", h["_id"]),
         "score": h["_score"],
         "title": h["_source"].get("title", ""),
         "content": h["_source"].get("content", "")[:200]}
        for h in resp["hits"]["hits"]
    ]

# 테스트
test_q = get_query(eval_rows[0])
print(f"테스트 쿼리: {test_q}")
results = search_sparse(test_q, top_k=5)
for r in results:
    print(f"  [{r['score']:.4f}] {r['docid']} | {r['title'][:60]}")

### 1-B: Dense 검색 (임베딩 기반)

In [ ]:
# ── Dense 임베딩 검색 ──────────────────────────────────────
# sentence-transformers 또는 OpenAI 임베딩 사용
# 서버에 sentence-transformers==2.6.1 설치되어 있음

DENSE_MODEL_NAME = "BAAI/bge-m3"  # 또는 "snunlp/KR-ELECTRA-discriminator"
DENSE_INDEX = "science_rag_dense"  # KNN 인덱스 (별도 구축 필요)

try:
    from sentence_transformers import SentenceTransformer
    dense_model = SentenceTransformer(DENSE_MODEL_NAME)
    print(f"Dense 모델 로드: {DENSE_MODEL_NAME} ✅")
    DENSE_AVAILABLE = True
except Exception as e:
    print(f"Dense 모델 로드 실패: {e}")
    print("→ DENSE_MODEL_NAME을 서버에 존재하는 모델로 변경하거나 별도 다운로드 필요")
    DENSE_AVAILABLE = False

def embed_query(query: str) -> Optional[List[float]]:
    if not DENSE_AVAILABLE: return None
    vec = dense_model.encode(query, normalize_embeddings=True)
    return vec.tolist()

def search_dense(query: str, top_k: int = 10) -> List[Dict]:
    """ES knn 검색 (knn 인덱스가 구축된 경우)"""
    vec = embed_query(query)
    if vec is None: return []
    dsl = {
        "size": top_k,
        "knn": {
            "field": "embedding",
            "query_vector": vec,
            "k": top_k,
            "num_candidates": top_k * 5,
        },
        "_source": ["docid", "title", "content"],
    }
    try:
        resp = es.search(index=DENSE_INDEX, body=dsl)
        return [
            {"docid": h["_source"].get("docid", h["_id"]),
             "score": h["_score"],
             "title": h["_source"].get("title", ""),
             "content": h["_source"].get("content", "")[:200]}
            for h in resp["hits"]["hits"]
        ]
    except NotFoundError:
        print(f"인덱스 {DENSE_INDEX} 없음 – 먼저 Dense 인덱스 구축 필요")
        return []

### 1-C: Hybrid 검색 (Sparse + Dense RRF)

In [ ]:
def rrf_fuse(
    result_lists: List[List[Dict]],
    k: int = 60,
    weights: Optional[List[float]] = None,
) -> List[Dict]:
    """
    Reciprocal Rank Fusion
    result_lists: 각 검색기의 결과 리스트 (docid 포함 dict)
    """
    if weights is None:
        weights = [1.0] * len(result_lists)
    score: Dict[str, float] = defaultdict(float)
    meta: Dict[str, Dict]   = {}

    for lst, w in zip(result_lists, weights):
        for rank, doc in enumerate(lst, start=1):
            did = str(doc["docid"])
            score[did] += w * (1.0 / (k + rank))
            if did not in meta:
                meta[did] = doc

    ranked = sorted(score.items(), key=lambda x: -x[1])
    return [{**meta[did], "rrf_score": sc} for did, sc in ranked]

def search_hybrid(
    query: str,
    top_k: int = 10,
    sparse_weight: float = 0.6,
    dense_weight: float = 0.4,
    phrase_boost: float = 2.0,
) -> List[Dict]:
    sparse_results = search_sparse(query, top_k=top_k * 2, phrase_boost=phrase_boost)
    dense_results  = search_dense(query, top_k=top_k * 2)

    if not dense_results:
        return sparse_results[:top_k]

    fused = rrf_fuse(
        [sparse_results, dense_results],
        weights=[sparse_weight, dense_weight]
    )
    return fused[:top_k]

# 테스트
hybrid = search_hybrid(test_q, top_k=5)
print(f"Hybrid 결과 ({test_q[:40]}...)")
for r in hybrid:
    print(f"  [{r.get('rrf_score', r.get('score', 0)):.4f}] {r['docid']} | {r['title'][:50]}")

---
## Phase 2: Reranker 실험

### 2-A: Fine-tuned Qwen3-Reranker-8B (LoRA)

In [ ]:
# ── Qwen3-Reranker 로드 (PEFT/LoRA 어댑터) ─────────────────
import torch

RERANKER_BASE = "Qwen/Qwen3-Reranker-8B"  # HuggingFace base
RERANKER_CKPT = MODEL_DIR  # fine-tuned 어댑터 경로

RERANKER_AVAILABLE = False
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from peft import PeftModel

    reranker_tokenizer = AutoTokenizer.from_pretrained(
        str(RERANKER_CKPT), trust_remote_code=True
    )
    reranker_base = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_BASE,
        num_labels=1,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    reranker_model = PeftModel.from_pretrained(reranker_base, str(RERANKER_CKPT))
    reranker_model.eval()
    print("Fine-tuned Qwen3-Reranker 로드 ✅")
    RERANKER_AVAILABLE = True
except Exception as e:
    print(f"Reranker 로드 실패: {e}")
    print("→ RERANKER_BASE 모델이 없거나 GPU 메모리 부족일 수 있음")

In [ ]:
def rerank(query: str, docs: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Fine-tuned Qwen3-Reranker로 docs 재정렬.
    docs: [{docid, title, content, ...}, ...]
    """
    if not RERANKER_AVAILABLE or not docs:
        return docs[:top_k]

    pairs = []
    for d in docs:
        passage = (d.get("title", "") + " " + d.get("content", "")).strip()
        pairs.append((query, passage))

    with torch.no_grad():
        enc = reranker_tokenizer(
            [p[0] for p in pairs],
            [p[1] for p in pairs],
            truncation=True,
            max_length=512,
            padding=True,
            return_tensors="pt",
        ).to(reranker_model.device)
        logits = reranker_model(**enc).logits.squeeze(-1).cpu().tolist()

    ranked = sorted(zip(docs, logits), key=lambda x: -x[1])
    return [{**d, "rerank_score": sc} for d, sc in ranked[:top_k]]

# ── Reranker 없을 때 fallback ──────────────────────────────
def score_based_rerank(query: str, docs: List[Dict], top_k: int = 3) -> List[Dict]:
    """BM25 점수 기반 단순 재정렬 (Reranker fallback)"""
    return sorted(docs, key=lambda d: -d.get("score", 0))[:top_k]

print("Reranker 함수 준비 완료 ✅")

---
## Phase 3: 앙상블 전략 실험

In [ ]:
# ── 기존 제출 파일들 로드 ─────────────────────────────────
SUB_FILES = {
    "m4":         SUB_DIR / "submission_m4.csv",
    "m4_chat":    SUB_DIR / "submission_m4_chatfilter.csv",
    "m4_dual":    SUB_DIR / "submission_m4_dualfuse.csv",
    "m6":         SUB_DIR / "submission_m6.csv",
    "m6_chat":    SUB_DIR / "submission_m6_dense_chatfilter.csv",
    "rrf3":       SUB_DIR / "submission_rrf_m4_m4f_m6f.csv",
    "phrase_v1":  SUB_DIR / "submission_sparse_strict_phraseboost_v1.csv",
}

subs: Dict[str, List[Dict]] = {}
sub_metas: Dict[str, Dict]  = {}

for name, path in SUB_FILES.items():
    if not path.exists():
        print(f"[SKIP] {name}: 파일 없음")
        continue
    rows, fmt = read_rows(path)
    id_col  = detect_col(list(rows[0].keys()), ["eval_id", "id", "query_id"])
    ref_col = detect_col(list(rows[0].keys()), ["references", "topk", "docids", "pred_docids"])
    subs[name] = rows
    sub_metas[name] = {"fmt": fmt, "id_col": id_col, "ref_col": ref_col, "path": path}
    print(f"[OK] {name}: {len(rows)}개 ({fmt})")

# 제출 파일 → {eval_id: [docid, ...]} 매핑
def to_pred_map(rows, meta) -> Dict[str, List[str]]:
    return {str(r[meta["id_col"]]).strip(): parse_docids(r.get(meta["ref_col"], []))
            for r in rows}

pred_maps: Dict[str, Dict[str, List[str]]] = {
    name: to_pred_map(subs[name], sub_metas[name]) for name in subs
}

# 베이스라인 대비 MAP@3 비교
if gt_map:
    print("\n--- MAP@3 비교 ---")
    for name, preds in pred_maps.items():
        score = map_at_3(preds, gt_map)
        print(f"  {name:20s}: {score:.4f}")

In [ ]:
# ── 제출 파일 간 top-3 일치율 비교 ────────────────────────
names = list(pred_maps.keys())
print("--- Top-3 일치율 행렬 ---")
eids_common = set.intersection(*[set(pred_maps[n].keys()) for n in names]) if names else set()

for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = names[i], names[j]
        same = sum(pred_maps[a].get(eid, [])[:3] == pred_maps[b].get(eid, [])[:3]
                   for eid in eids_common)
        n = len(eids_common)
        print(f"  {a:20s} vs {b:20s}: {same}/{n} = {same/n:.1%}")

In [ ]:
# ── 다중 파일 RRF 앙상블 ──────────────────────────────────
def ensemble_rrf(
    pred_maps_selected: Dict[str, Dict[str, List[str]]],
    weights: Optional[Dict[str, float]] = None,
    k: int = 60,
) -> Dict[str, List[str]]:
    """
    여러 제출 파일을 RRF로 앙상블
    """
    if weights is None:
        weights = {n: 1.0 for n in pred_maps_selected}

    all_eids = set().union(*[set(m.keys()) for m in pred_maps_selected.values()])
    result: Dict[str, List[str]] = {}

    for eid in all_eids:
        score: Dict[str, float] = defaultdict(float)
        for name, pmap in pred_maps_selected.items():
            docs = pmap.get(eid, [])
            w = weights.get(name, 1.0)
            for rank, docid in enumerate(docs[:10], start=1):
                score[docid] += w / (k + rank)
        result[eid] = [d for d, _ in sorted(score.items(), key=lambda x: -x[1])[:3]]

    return result

# ── 실험 1: 균등 가중치 RRF ───────────────────────────────
ens_uniform = ensemble_rrf(pred_maps)
if gt_map:
    print(f"RRF 균등앙상블 MAP@3: {map_at_3(ens_uniform, gt_map):.4f}")

# ── 실험 2: 성능 기반 가중치 RRF ─────────────────────────
if gt_map:
    sub_scores = {n: map_at_3(pred_maps[n], gt_map) for n in pred_maps}
    total_sc = sum(sub_scores.values()) or 1
    score_weights = {n: sc / total_sc for n, sc in sub_scores.items()}
    ens_weighted = ensemble_rrf(pred_maps, weights=score_weights)
    print(f"RRF 가중앙상블 MAP@3: {map_at_3(ens_weighted, gt_map):.4f}")
else:
    # GT 없을 때: 수동 가중치 (실험적으로 조정)
    manual_weights = {
        "m4":        1.2,
        "m4_chat":   1.0,
        "phrase_v1": 1.1,
        "m6_chat":   0.9,
    }
    ens_weighted = ensemble_rrf(
        {n: pred_maps[n] for n in manual_weights if n in pred_maps},
        weights=manual_weights
    )
    print(f"RRF 수동가중앙상블 생성: {len(ens_weighted)}개")

---
## Phase 4: 쿼리 라우터 실험

### 4-A: 규칙 기반 라우터 (개선)

In [ ]:
# ── 쿼리 분류: 과학 상식 vs 일반 대화 ─────────────────────
SCIENCE_KEYWORDS = [
    # 물리
    "에너지", "속도", "가속도", "질량", "온도", "압력", "전류", "전압", "전기", "자기",
    "빛", "파동", "열", "힘", "운동", "마찰", "중력", "진공", "원자", "핵",
    # 화학
    "원소", "분자", "화합물", "산화", "환원", "촉매", "ph", "산성", "염기", "화학식",
    "반응", "용해", "증발", "응결", "융해", "결정", "이온", "전자",
    # 생물
    "세포", "유전자", "dna", "광합성", "호흡", "진화", "생태계", "생물", "동물", "식물",
    "미생물", "바이러스", "단백질", "효소", "뉴런", "면역",
    # 지구과학/우주
    "행성", "지층", "지진", "화산", "대기", "기후", "태양", "달", "별", "은하",
    "소행성", "블랙홀", "우주", "중력파", "대륙", "해류",
    # 수학/측정
    "측정", "단위", "mol", "밀도", "농도", "파장", "주파수", "진폭",
]

CHAT_KEYWORDS = [
    "안녕", "고마워", "감사", "추천", "도와줘", "뭐야", "누구야",
    "오늘", "날씨", "기분", "맛집", "여행", "쇼핑", "요리", "영화",
    "음악", "게임", "취미", "친구", "가족",
]

COUNTRY_COMPARE_HINTS = [
    "각 나라", "나라별", "국가별", "여러 나라", "국가 간", "비교", "현황",
    "통계", "순위", "oecd", "gdp 대비", "지출", "비율",
]

def classify_query(query: str) -> str:
    """
    쿼리 분류:
    - 'science': 과학 상식 → ES 검색 필요
    - 'stat':    통계/국가비교 → Sparse 강화 필요
    - 'chat':    일반 대화 → 검색 생략 가능
    """
    q = query.lower()
    n_sci  = sum(1 for k in SCIENCE_KEYWORDS if k in q)
    n_chat = sum(1 for k in CHAT_KEYWORDS if k in q)
    n_stat = sum(1 for k in COUNTRY_COMPARE_HINTS if k in q)

    if len(q) <= 10 or (n_chat > 0 and n_sci == 0):
        return "chat"
    if n_stat >= 2 or any(h in q for h in ["각 나라", "나라별"]):
        return "stat"
    return "science"

# 테스트
test_queries = [
    "광합성이 일어나는 과정을 설명해줘",
    "각 나라에서의 공교육 지출 현황에 대해 알려줘.",
    "안녕하세요! 오늘 날씨 어때요?",
    "DNA의 구조와 기능은?",
]
for q in test_queries:
    print(f"  [{classify_query(q):8s}] {q}")

In [ ]:
# ── 쿼리 유형별 검색 전략 라우팅 ─────────────────────────
def routed_search(
    query: str,
    top_k: int = 10,
    use_reranker: bool = True,
) -> List[Dict]:
    qtype = classify_query(query)

    if qtype == "chat":
        # 일반 대화: 짧은 Sparse만
        docs = search_sparse(query, top_k=top_k, use_synonym=False, phrase_boost=1.0)

    elif qtype == "stat":
        # 통계/비교: 동의어 확장 + phrase boost 강화
        docs = search_sparse(query, top_k=top_k * 2, use_synonym=True, phrase_boost=3.0)
        # Hybrid 시도
        if DENSE_AVAILABLE:
            dense = search_dense(query, top_k=top_k)
            docs  = rrf_fuse([docs, dense], weights=[0.7, 0.3])[:top_k * 2]

    else:  # science
        # 과학 질문: Hybrid
        docs = search_hybrid(query, top_k=top_k * 2, sparse_weight=0.55, dense_weight=0.45)

    # Reranking
    if use_reranker and RERANKER_AVAILABLE:
        docs = rerank(query, docs, top_k=top_k)
    else:
        docs = docs[:top_k]

    return docs

print("라우터 함수 준비 완료 ✅")

### 4-B: 학습형 메타 라우터 (sklearn)

In [ ]:
import re as _re
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

NUM_RE    = _re.compile(r"\d")
ENG_RE    = _re.compile(r"[A-Za-z]")
HANGUL_RE = _re.compile(r"[가-힣]")

def build_features(query: str, candidates: Dict[str, List[str]]) -> List[float]:
    """쿼리 + 후보 결과 특징 벡터 생성"""
    q    = query.strip()
    toks = q.split()
    feats = [
        float(len(q)),
        float(len(toks)),
        float(len(NUM_RE.findall(q))),
        float(len(ENG_RE.findall(q))),
        float(len(HANGUL_RE.findall(q))),
        1.0 if "?" in q else 0.0,
        float(sum(1 for k in SCIENCE_KEYWORDS if k in q.lower())),
        float(sum(1 for k in CHAT_KEYWORDS    if k in q.lower())),
        float(sum(1 for k in COUNTRY_COMPARE_HINTS if k in q.lower())),
        1.0 if len(q) <= 12 else 0.0,
    ]
    # 후보 간 Jaccard 유사도 특징
    cnames = sorted(candidates.keys())
    for i in range(len(cnames)):
        for j in range(i+1, len(cnames)):
            sa = set(candidates[cnames[i]][:3])
            sb = set(candidates[cnames[j]][:3])
            union = len(sa | sb) or 1
            feats.append(len(sa & sb) / union)
    return feats

# ── 학습 데이터 준비 (GT 있는 경우) ─────────────────────────
if gt_map and len(pred_maps) >= 2:
    X_train, y_train = [], []
    for eid in gt_map:
        row   = eval_map.get(eid)
        if not row: continue
        query = get_query(row)
        cands = {n: pred_maps[n].get(eid, []) for n in pred_maps}

        # 각 후보의 AP@3으로 최우수 전략 레이블
        best_name, best_ap = None, -1.0
        for name in pred_maps:
            ap = ap_at_3(pred_maps[name].get(eid, []), gt_map.get(eid, []))
            if ap > best_ap:
                best_ap, best_name = ap, name

        if best_name:
            X_train.append(build_features(query, cands))
            y_train.append(best_name)

    print(f"학습 샘플: {len(X_train)}개")
    print("레이블 분포:", Counter(y_train))

    le = LabelEncoder()
    y_enc = le.fit_transform(y_train)

    # 모델 비교
    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
        "GBM":          GradientBoostingClassifier(n_estimators=100, random_state=42),
        "LR":           LogisticRegression(max_iter=500, random_state=42),
    }
    print("\n--- CV 정확도 ---")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for name, clf in models.items():
        scores = cross_val_score(clf, X_train, y_enc, cv=cv, scoring="accuracy")
        print(f"  {name:20s}: {scores.mean():.3f} ± {scores.std():.3f}")
else:
    print("GT 없음 – 학습형 라우터 생략. 규칙 기반 라우터 사용.")

---
## Phase 5: 전체 파이프라인 실행 & 제출 파일 생성

In [ ]:
# ── 전체 실험 파이프라인 ───────────────────────────────────
from tqdm import tqdm

EXPERIMENT_NAME = "hybrid_rerank_routed_v1"  # ← 실험마다 변경
OUTPUT_PATH = SUB_DIR / f"submission_{EXPERIMENT_NAME}.csv"

def run_pipeline(
    eval_rows: List[Dict],
    strategy: str = "hybrid",  # "sparse" | "hybrid" | "ensemble" | "routed"
    use_reranker: bool = True,
    top_k_retrieve: int = 20,
    top_k_final: int = 3,
    ensemble_preds: Optional[Dict[str, List[str]]] = None,
) -> List[Dict]:
    """
    전체 RAG 파이프라인 실행
    strategy:
      - 'sparse'   : BM25 + phrase boost + 동의어
      - 'hybrid'   : Sparse + Dense RRF
      - 'routed'   : 쿼리 유형별 최적 전략
      - 'ensemble' : 기존 제출 파일 앙상블 (ensemble_preds 필요)
    """
    out_rows = []
    errors   = []

    for row in tqdm(eval_rows, desc=f"[{strategy}]"):
        eid   = str(row[EVAL_ID_COL]).strip()
        query = get_query(row)

        try:
            if strategy == "ensemble" and ensemble_preds:
                top3 = ensemble_preds.get(eid, [])

            elif strategy == "routed":
                docs = routed_search(query, top_k=top_k_retrieve, use_reranker=use_reranker)
                top3 = [d["docid"] for d in docs[:top_k_final]]

            elif strategy == "hybrid":
                docs = search_hybrid(query, top_k=top_k_retrieve)
                if use_reranker and RERANKER_AVAILABLE:
                    docs = rerank(query, docs, top_k=top_k_final)
                top3 = [d["docid"] for d in docs[:top_k_final]]

            else:  # sparse
                docs = search_sparse(query, top_k=top_k_retrieve, use_synonym=True)
                if use_reranker and RERANKER_AVAILABLE:
                    docs = rerank(query, docs, top_k=top_k_final)
                top3 = [d["docid"] for d in docs[:top_k_final]]

        except Exception as e:
            top3 = base_preds.get(eid, [])  # 실패 시 베이스라인 사용
            errors.append({"eval_id": eid, "error": str(e)})

        # 출력 행 구성
        out_row = dict(row)
        out_row["references"] = top3
        out_rows.append(out_row)

    print(f"완료: {len(out_rows)}개, 에러: {len(errors)}개")
    return out_rows, errors

print("파이프라인 함수 준비 완료 ✅")
print(f"실험 이름: {EXPERIMENT_NAME}")
print(f"출력 경로: {OUTPUT_PATH}")

In [ ]:
# ── 실험 실행 ─────────────────────────────────────────────
# ※ 전략 선택: strategy 변수만 바꾸면 됨

STRATEGY = "ensemble"   # "sparse" | "hybrid" | "routed" | "ensemble"

if STRATEGY == "ensemble":
    # 앙상블 결과 사용 (ES 불필요, 빠름)
    out_rows, errors = run_pipeline(
        eval_rows,
        strategy="ensemble",
        ensemble_preds=ens_weighted,
    )
else:
    # 실시간 ES 검색
    out_rows, errors = run_pipeline(
        eval_rows,
        strategy=STRATEGY,
        use_reranker=RERANKER_AVAILABLE,
        top_k_retrieve=20,
        top_k_final=3,
    )

# 저장
write_rows(OUTPUT_PATH, out_rows, "jsonl")
print(f"\n✅ 저장 완료: {OUTPUT_PATH}")
print(f"   sha256: {sha256_file(OUTPUT_PATH)}")
print(f"   크기:   {OUTPUT_PATH.stat().st_size:,} bytes")

In [ ]:
# ── MAP@3 평가 (GT 있는 경우) ─────────────────────────────
new_preds = {
    str(r[EVAL_ID_COL]).strip(): parse_docids(r.get("references", []))
    for r in out_rows
}

if gt_map:
    new_score = map_at_3(new_preds, gt_map)
    base_score = map_at_3(base_preds, gt_map)
    delta = new_score - base_score
    print(f"\n{'='*40}")
    print(f"  베이스라인 MAP@3: {base_score:.4f}")
    print(f"  새 실험   MAP@3: {new_score:.4f}")
    print(f"  개선폭:         {delta:+.4f}")
    print(f"{'='*40}")
else:
    print("GT 없음 – 제출 후 리더보드 점수 확인 필요")

# 에러 케이스 확인
if errors:
    print(f"\n에러 {len(errors)}개:")
    for e in errors[:5]:
        print(f"  eval_id={e['eval_id']}: {e['error']}")

---
## Phase 6: 오류 분석 & 개선 아이디어

In [ ]:
# ── 쿼리별 오류 분석 ──────────────────────────────────────
if gt_map:
    analysis = []
    for eid, gt_docs in gt_map.items():
        pred_docs  = new_preds.get(eid, [])
        base_docs  = base_preds.get(eid, [])
        ap_new     = ap_at_3(pred_docs, gt_docs)
        ap_base    = ap_at_3(base_docs, gt_docs)
        query      = get_query(eval_map.get(eid, {}))
        qtype      = classify_query(query)
        analysis.append({
            "eval_id":    eid,
            "query":      query[:80],
            "qtype":      qtype,
            "ap_base":    ap_base,
            "ap_new":     ap_new,
            "delta":      ap_new - ap_base,
            "pred":       pred_docs[:3],
            "gt":         gt_docs[:3],
        })

    df_analysis = pd.DataFrame(analysis).sort_values("delta")

    print("📉 개악된 케이스 (delta < 0) TOP-10:")
    print(df_analysis[df_analysis.delta < 0][["eval_id","query","qtype","ap_base","ap_new","delta"]].head(10).to_string())

    print("\n📈 개선된 케이스 (delta > 0) TOP-10:")
    print(df_analysis[df_analysis.delta > 0][["eval_id","query","qtype","ap_base","ap_new","delta"]].tail(10).to_string())

    print("\n쿼리 유형별 평균 AP@3:")
    print(df_analysis.groupby("qtype")[["ap_base","ap_new","delta"]].mean())

In [ ]:
# ── 실험 결과 요약 저장 ───────────────────────────────────
report = {
    "experiment":   EXPERIMENT_NAME,
    "strategy":     STRATEGY,
    "use_reranker": RERANKER_AVAILABLE,
    "dense_model":  DENSE_MODEL_NAME if DENSE_AVAILABLE else None,
    "n_total":      len(out_rows),
    "n_errors":     len(errors),
    "output":       str(OUTPUT_PATH),
    "sha256":       sha256_file(OUTPUT_PATH),
    "map_at3_base": map_at_3(base_preds, gt_map) if gt_map else None,
    "map_at3_new":  map_at_3(new_preds, gt_map)  if gt_map else None,
    "timestamp":    time.strftime("%Y-%m-%dT%H:%M:%S"),
}

report_path = SUB_DIR / f"report_{EXPERIMENT_NAME}.json"
with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print(f"\n📄 리포트 저장: {report_path}")

---
## Phase 7: 고득점 체크리스트 & 다음 실험 아이디어

```
✅ 완료된 실험
─────────────────────────────────────────────────────
[ ] Phase 1-A: Sparse BM25 + phrase boost + 동의어 확장
[ ] Phase 1-B: Dense KNN 검색 (임베딩)
[ ] Phase 1-C: Hybrid RRF (Sparse + Dense)
[ ] Phase 2-A: Fine-tuned Qwen3-Reranker-8B
[ ] Phase 3:   다중 제출 파일 RRF 앙상블
[ ] Phase 4-A: 쿼리 유형 분류 라우터
[ ] Phase 4-B: 학습형 메타 라우터 (sklearn)

🔧 다음 실험 아이디어
─────────────────────────────────────────────────────
1. 동의어 사전 확장
   - artifacts/science_synonyms.txt 보강
   - 도메인 특화 용어 추가 (교과서 기반)

2. 쿼리 리라이팅 (Query Rewriting)
   - LLM으로 쿼리를 검색 최적화 형태로 변환
   - "각 나라의 공교육 지출" → "OECD 공교육 예산 비율 GDP"

3. HyDE (Hypothetical Document Embedding)
   - 쿼리에 대한 가상 답변을 생성 후 그 답변으로 검색

4. Reranker 체크포인트 비교
   - checkpoint-1312 vs checkpoint-1968 vs 최종 어댑터

5. 앙상블 가중치 최적화
   - Optuna로 RRF 가중치 자동 탐색

6. 통계/비교 쿼리 특화 처리
   - stat_compare_sparse_override 로직 개선
   - 국가명 NER 추출 후 쿼리 보강
```

In [ ]:
# ── 보너스: Optuna로 RRF 가중치 자동 최적화 ─────────────
# GT 있을 때만 실행

if gt_map and len(pred_maps) >= 2:
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        CANDIDATES = list(pred_maps.keys())

        def objective(trial):
            weights = {n: trial.suggest_float(n, 0.1, 3.0) for n in CANDIDATES}
            ens = ensemble_rrf(pred_maps, weights=weights)
            return map_at_3(ens, gt_map)

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=100, show_progress_bar=True)

        print(f"\n최적 MAP@3: {study.best_value:.4f}")
        print("최적 가중치:")
        for k, v in study.best_params.items():
            print(f"  {k}: {v:.3f}")

        # 최적 가중치로 제출 파일 생성
        best_ens = ensemble_rrf(pred_maps, weights=study.best_params)
        best_rows = []
        for row in eval_rows:
            eid = str(row[EVAL_ID_COL]).strip()
            nr = dict(row)
            nr["references"] = best_ens.get(eid, [])
            best_rows.append(nr)
        opt_path = SUB_DIR / "submission_optuna_rrf.csv"
        write_rows(opt_path, best_rows, "jsonl")
        print(f"\n✅ Optuna 앙상블 저장: {opt_path}")

    except ImportError:
        print("optuna 미설치 – pip install optuna --quiet 실행 후 재시도")
else:
    print("GT 없음 또는 제출 파일 1개 이하 – Optuna 최적화 스킵")